# Alternative parametrizations

`GaugeFixer` was initially designed for gauge fixing within the family of generalized one-hot models. However, its fast algorithms can also be used to transform to and from an alternative class of models in which genotypes are encoded with proper, Kronecker-factorizable basis vectors, whose coefficients have distinct interpretations.

Let $f$ be the $\alpha^L$-dimensional vector representing an arbitrary sequence-function landscape, and let $\beta = Af$ denote a set of coefficients describing that landscape. If $A=\bigotimes_{l=1}^L A_l$ with each $A_l \in \mathbb{R}^{\alpha \times \alpha}$, then any coefficient vector $\beta$ for a hierarchical model can be computed efficiently from generalized one-hot parameters $\theta$, since $f=X\theta$:

$$
\beta = Af = AX\theta
= \left(\bigotimes_{l=1}^L A_l\right)\left(\bigotimes_{l=1}^L X_l\right)\theta
= \left(\bigotimes_{l=1}^L A_l X_l\right)\theta.
$$

Moreover, as long as each $A_l$ is invertible, we can directly define a basis parameterization of the sequence-function model from the associated coefficients:

$$
f = A^{-1}\beta
= \left(\bigotimes_{l=1}^{L} A_l^{-1}\right)\beta.
$$

These algorithms are, in principle, applicable to any site-factorizable basis. In practice, `GaugeFixer` currently provides built-in support for specific bases proposed in the literature, exposed through `set_basis_coeffs` and `get_basis_coeffs` via basis names:

- **Walsh-Hadamard coefficients** (`'WH'`, limited to binary sequences): coefficients associated to this orthonormal basis represent the normalized spectral contrasts associated to mutational effects and (pairwise and higher-order) epistatic effects relative to a wild-type reference sequence `wt_seq` ([Weinberger (1991)](https://dl.acm.org/doi/10.1007/BF00216965)).

$$
A_l^{-1} = \frac{1}{\sqrt{2}}
\begin{bmatrix}
1 & 1 \\
1 & -1
\end{bmatrix}
$$

- **Evolutionary Walsh-Hadamard coefficients** (`'eWH'`, limited to binary sequences): coefficients in this basis represent the normalized spectral contrasts associated to mutational effecits and (pairwise and higher-order) epistatic effects relative to a wild-type reference sequence `wt_seq` and with respect to a probability measure defined by a factorizable probability distribution $\pi_l^c$ ($\pi_l^{c_1} = 1-p_l$ and $\pi_l^{c_2} = p_l$ relative to the parametrization presented by [Tsui et al. (2026)](https://www.biorxiv.org/content/10.64898/2026.07.08.737351v1.full.pdf)). It reduces to the classical Walsh-Hadamard basis when $\pi_l^c = 0.5$, up to normalization.

$$
A_l^{-1} =
\begin{bmatrix}
1 & \frac{\pi_l^{c_2}}{\sqrt{\pi_l^{c_1} \pi_l^{c_2}}} \\
1 & -\frac{\pi_l^{c_1}}{\sqrt{\pi_l^{c_1} \pi_l^{c_2}}}
\end{bmatrix}
$$

- **Background-averaged epistatic coefficients** (`'background-averaged'`): coefficients in this basis represent average mutational effects (and higher-order epistatic terms) relative to a wild-type reference sequence `wt_seq`, averaged over all genetic backgrounds. In the biallelic case, this reduces to the classical Walsh–Hadamard basis for uniform `pi_lc` and to the evolutionary Walsh-Hadamard basis for non-uniform distributions. Thus, it provides a generalization to the multi-allelic setting ([Faure et al. (2024)](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1012132)) and to non-uniform background distributions specified by `pi_lc`, up to normalization constants. For one site, with allele \(1\) chosen as wild type:

$$
A_l =
\begin{bmatrix}
\pi_l^{c_1} & \pi_l^{c_2} & \pi_l^{c_3} & \cdots & \pi_l^{c_{\alpha}} \\
-1 & 1 & 0 & \cdots & 0 \\
-1 & 0 & 1 & \cdots & 0 \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
-1 & 0 & 0 & \cdots & 1
\end{bmatrix}
$$

- **Fourier coefficients** (`'fourier'`): coefficients in this basis correspond to a specific orthonormal Fourier basis at each site, defined relative to a wild-type sequence `wt_seq` ([Brookes et al. (2022)](https://www.pnas.org/doi/10.1073/pnas.2109649118)). In the biallelic case, this also reduces to the normalized Walsh–Hadamard basis ([Weinberger (1991)](https://dl.acm.org/doi/10.1007/BF00216965)) and in the 4-allele case reduces to the tetrahedral encoding proposed by [Stormo (2010)](https://doi.org/10.1534/genetics.110.126052) for DNA sequences after normalization. Because $A_l^{-1}$ is orthonormal, computing the inverse is straightforward: $A_l=(A_l^{-1})^T$. Let $e_i$ be the indicator vector of the wild-type allele, and define $w=\mathbf{1}_{\alpha}-\sqrt{\alpha}\,e_i$. Then the site-specific basis is

$$
A_l^{-1} = I_{\alpha} - \frac{2ww^T}{\lVert w\rVert_2^2}.
$$



In [1]:
import numpy as np

from gaugefixer import AllOrderModel

### Initializing a model

The first thing we always need is to define a model. Here for simplicity, we use an `AllOrderModel` model with randomly defined parameters, but it is applicable to any hierarchical model available in `GaugeFixer` with user-defined parameters. 

In [2]:
model = AllOrderModel(L=3, alphabet_name="dna")
model.set_random_params()
model

AllOrderModel(L=3,alphabet_name=dna,n_features=125,n_orbits=8)

In [3]:
f1 = model.get_landscape()

### Computing background-averaged epistasis coefficients

To compute the background averaged epistasis coefficients we simply need to specify the reference or wild-type sequence `wt_seq`. Note that the output `pd.Series` is now indexed by a set of positions and combination of mutations at those positions rather than subsequences as the parameters of the generalized one-hot model. 


In [4]:
coeffs = model.get_basis_coeffs(basis_name='background-averaged', wt_seq='ACG')
coeffs

((), )                      2.140726
((0,), A>C)                -0.510235
((0,), A>G)                -1.380427
((0,), A>T)                -0.705175
((1,), C>A)                 0.815750
                              ...   
((0, 1, 2), A>T;C>G;G>C)    3.053489
((0, 1, 2), A>T;C>G;G>T)   -1.767025
((0, 1, 2), A>T;C>T;G>A)   -5.574246
((0, 1, 2), A>T;C>T;G>C)   -5.680209
((0, 1, 2), A>T;C>T;G>T)   -6.197801
Length: 64, dtype: float64

By default, the coefficients will represent averages across genetic backgrounds uniformly distributed, but they can also be averaged across arbitrary site-factorizable distributions defined by `pi_lc`.

In [5]:
pi_lc = [np.array([0.1, 0.2, 0.3, 0.4]),
         np.array([0.2, 0.2, 0.3, 0.3]), 
         np.array([0.4, 0.1, 0.1, 0.4])]
model.get_basis_coeffs(basis_name="background-averaged", wt_seq="ACG", pi_lc=pi_lc)

((), )                      2.071623
((0,), A>C)                -1.824709
((0,), A>G)                -2.211188
((0,), A>T)                -2.080554
((1,), C>A)                 0.808773
                              ...   
((0, 1, 2), A>T;C>G;G>C)    3.053489
((0, 1, 2), A>T;C>G;G>T)   -1.767025
((0, 1, 2), A>T;C>T;G>A)   -5.574246
((0, 1, 2), A>T;C>T;G>C)   -5.680209
((0, 1, 2), A>T;C>T;G>T)   -6.197801
Length: 64, dtype: float64

### Computing Fourier coefficients

To compute the Fourier coefficients we simply need to specify the reference or wild-type sequence `wt_seq` and, as before, we will get a `pd.Series` indexed by a set of positions and combination of mutations at those positions relative to the chosen wild-type sequence `wt_seq`

In [6]:
model.get_basis_coeffs(basis_name="fourier", wt_seq="ACG")

((), )                      17.125806
((0,), A>C)                  3.150733
((0,), A>G)                 -0.330034
((0,), A>T)                  2.370974
((1,), C>A)                 -0.043254
                              ...    
((0, 1, 2), A>T;C>G;G>C)     0.226875
((0, 1, 2), A>T;C>G;G>T)    -2.168041
((0, 1, 2), A>T;C>T;G>A)     1.129874
((0, 1, 2), A>T;C>T;G>C)    -1.401208
((0, 1, 2), A>T;C>T;G>T)     0.864620
Length: 64, dtype: float64

### Defining models through the alternative parametrizations

In addition to computation of the associated coefficients from an arbitrary generalized one-hot hierarchical model, we can also define models via coefficients in these bases that were previously estimated using the `set_basis_coeffs` method. For instance, lets imagine we have a set of background-averaged coefficients as a `pd.Series` in the variable `coeffs` for the wild-type sequence `ACG` and want to define the corresponding model

In [7]:
coeffs

((), )                      2.140726
((0,), A>C)                -0.510235
((0,), A>G)                -1.380427
((0,), A>T)                -0.705175
((1,), C>A)                 0.815750
                              ...   
((0, 1, 2), A>T;C>G;G>C)    3.053489
((0, 1, 2), A>T;C>G;G>T)   -1.767025
((0, 1, 2), A>T;C>T;G>A)   -5.574246
((0, 1, 2), A>T;C>T;G>C)   -5.680209
((0, 1, 2), A>T;C>T;G>T)   -6.197801
Length: 64, dtype: float64

In [8]:
model = AllOrderModel(L=3, alphabet_name="dna")
model.set_basis_coeffs(coeffs=coeffs, basis_name="background-averaged", wt_seq="ACG")

We can verify that this encodes exactly the same model as before

In [9]:
f2 = model.get_landscape()
np.allclose(f1, f2)

True

and compute the background averaged epistasis coefficients relative to a different wild-type sequence e.g. `TGA`

In [10]:
model.get_basis_coeffs(basis_name="background-averaged", wt_seq="TGA")

((), )                      2.140726
((0,), T>A)                 0.705175
((0,), T>C)                 0.194940
((0,), T>G)                -0.675252
((1,), G>A)                 1.217741
                              ...   
((0, 1, 2), T>G;G>C;A>G)   -1.109583
((0, 1, 2), T>G;G>C;A>T)   -0.881436
((0, 1, 2), T>G;G>T;A>C)    5.156030
((0, 1, 2), T>G;G>T;A>G)   -2.961225
((0, 1, 2), T>G;G>T;A>T)   -0.599100
Length: 64, dtype: float64

or calculate the gauge-fixed coefficients in the `zero-sum` gauge for the corresponding generalized one-hot model

In [11]:
model.get_fixed_params(gauge='zero-sum')

((), )              2.140726
((0,), A)           0.648959
((0,), C)           0.138724
((0,), G)          -0.731468
((0,), T)          -0.056216
                      ...   
((0, 1, 2), TGT)   -1.061137
((0, 1, 2), TTA)   -0.438287
((0, 1, 2), TTC)   -1.574907
((0, 1, 2), TTG)    1.943813
((0, 1, 2), TTT)    0.069381
Length: 125, dtype: float64